# 01 — Dữ liệu và kiểm tra chất lượng

Notebook này kiểm tra bộ dữ liệu sạch dùng trong nghiên cứu: phạm vi mẫu, độ phủ lợi suất, quy tắc point-in-time và các đối soát chất lượng. Dữ liệu đầu vào là artefact đã được pipeline tạo ra, vì vậy notebook có thể chạy offline và không gọi API bên ngoài.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), f'Không tìm thấy repository root từ {Path.cwd()}'
OUT = ROOT / 'outputs' / 'vn_period_factors'
ROOT

In [ ]:
returns = pd.read_csv(OUT / 'vn100_returns_clean.csv', parse_dates=['date'])
factors = pd.read_csv(OUT / 'vn100_factors_monthly.csv', parse_dates=['Date'])
audit = pd.read_csv(OUT / 'vn100_formation_audit.csv', parse_dates=['formation_date'])
quality = pd.read_csv(OUT / 'vn100_data_quality.csv')
reconciliation = pd.read_csv(OUT / 'vn100_return_reconciliation.csv')

overview = pd.Series({
    'return_rows': len(returns),
    'tickers': returns['ticker'].nunique(),
    'return_start': returns['date'].min().date(),
    'return_end': returns['date'].max().date(),
    'formation_dates': audit['formation_date'].nunique(),
    'factor_rows': len(factors),
    'factor_start': factors['Date'].min().date(),
    'factor_end': factors['Date'].max().date(),
}, name='value').to_frame()
display(overview)
display(returns['return_source'].value_counts(dropna=False).rename('rows').to_frame())

## Kiểm tra point-in-time

Mỗi kỳ formation phải dùng dữ liệu kế toán đã có thể quan sát tại thời điểm đó. Pipeline lưu `information_date` và cờ `look_ahead` để kiểm tra trực tiếp điều kiện này.

In [ ]:
audit['information_date'] = pd.to_datetime(audit['information_date'], errors='coerce')
pit_checks = {
    'information_date_after_formation': int((audit['information_date'] > audit['formation_date']).sum()),
    'look_ahead_flagged': int(audit['look_ahead'].fillna(False).astype(bool).sum()),
}
pit_checks['duplicate_ticker_formation'] = int(audit.duplicated(['formation_date', 'ticker']).sum())
pit_checks['invalid_simple_returns'] = int((returns['return'] <= -1).sum())
display(pd.Series(pit_checks, name='violations').to_frame())
assert not any(pit_checks.values()), pit_checks

## Độ phủ nhân tố và báo cáo chất lượng

In [ ]:
factor_cols = ['MKT_RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']
coverage = pd.DataFrame({
    'months': factors[factor_cols].notna().sum(),
    'first_month': {c: factors.loc[factors[c].notna(), 'Date'].min() for c in factor_cols},
    'last_month': {c: factors.loc[factors[c].notna(), 'Date'].max() for c in factor_cols},
})
display(coverage)
display(quality)
reconciliation['abs_gap'] = reconciliation['reported_minus_price_return'].abs()
display(reconciliation.groupby('return_source', dropna=False).agg(rows=('ticker', 'size'), mean_abs_gap=('abs_gap', 'mean'), max_abs_gap=('abs_gap', 'max')))
display(reconciliation.nlargest(10, 'abs_gap')[['period', 'ticker', 'date', 'return_source', 'reported_minus_price_return']])

## Tái tạo dữ liệu

Để dựng lại artefact nguồn, đặt hai ZIP đầu vào đúng tên trong `data/raw/`, sau đó chạy lệnh baseline được ghi trong README. Notebook này cố ý chỉ đọc kết quả đã version hóa để tránh vô tình ghi đè output nghiên cứu.